In [ ]:
# Environment setup: pick up local edits to libdpy/ without restarting the kernel,
# and install the public library when missing (e.g. on Google Colab).
# autoreload is only a local-dev convenience (live-reloading libdpy edits). Skip it if the
# extension can't load - e.g. on Colab's Python 3.12, where IPython's autoreload still imports
# the removed `imp` module and raises ModuleNotFoundError.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

try:
    import libdpy
except ImportError:
    %pip install -q "libdpy @ git+https://github.com/applied-dp-course/pub_lib.git"
    import libdpy

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

from libdpy.hypothesis_testing import FPR_from_threshold, TPR_from_threshold
from libdpy.attacks.membership_inference.classical_auditing_utils import threshold_guesser
from libdpy.attacks.membership_inference.classical_auditing_utils import epsilon_lower_bound
from libdpy.privacy_mechanisms.noise import Laplace_sum
from libdpy.visualization.privacy_plots import PrivacyPlot
from libdpy.assignment_specific.privacy_auditing.visualizations import (
    binomial_alpha_bounds_interactive,
    binomial_beta_bounds_interactive,
)
from libdpy.visualization.roc_plots import EmpROCVisualizer, TheoryROCVisualizer

# Privacy Auditing

In the previous lectures we *designed* private mechanisms and *proved* upper bounds on how much they leak. Auditing asks the opposite, empirical question:

> Given only black-box access to a mechanism, **how much privacy loss can we actually demonstrate?**

We put ourselves in the shoes of an adversary: run the mechanism many times, try to guess whether a target record was present, and turn our guessing success into a **lower bound** on $\varepsilon$. This is the workhorse behind finding bugs in DP implementations and behind empirical privacy claims for systems like DP-SGD.

This notebook builds the idea up in stages:
1. Recap the hypothesis-testing view of DP and the $(\varepsilon, \delta)$ trade-off bound.
2. Meet a family of noise distributions; for each, write down its PDF, CDF, and privacy loss, and look at its ROC.
3. Meet a noise distribution with **no closed form** (the difference of two lognormals) and estimate its ROC by **sampling**.
4. Run the audit empirically by playing the adversary many times.
5. Ask whether a single run gives an *exact* answer (it does not) and use **tail bounds** to turn a noisy estimate into a *valid* high-confidence lower bound.

## Recap: DP as hypothesis testing

In the previous lecture we recast differential privacy as a **binary hypothesis test**. An adversary observes the mechanism's output and must decide between

$$H_0:\ \text{the target record is absent (negative)} \qquad H_1:\ \text{the target record is present (positive)},$$

which correspond to running the mechanism on two neighboring datasets. A decision rule is summarised by its **false positive rate** $\mathrm{FPR}$ and **true positive rate** $\mathrm{TPR}$, and the best achievable trade-off traces out the **ROC curve**. The closer that curve hugs the diagonal $\mathrm{TPR}=\mathrm{FPR}$, the harder the adversary's job and the more private the mechanism.

The key bridge to DP: a mechanism is $(\varepsilon, \delta)$-DP **if and only if** its ROC curve stays below the two symmetric lines

$$\mathrm{TPR} \le e^{\varepsilon}\,\mathrm{FPR} + \delta, \qquad \mathrm{FPR} \le e^{\varepsilon}\,\mathrm{TPR} + \delta .$$

So $\varepsilon$ controls the *slope* near the corners and $\delta$ the *intercept*. The plot below redraws this $(\varepsilon, \delta)$ bound (blue) against the ROC curves of a couple of noise mechanisms — the picture we ended the last lecture on.

In [ ]:
PrivacyPlot(sensitivity=1, std=1.5, distribution_types=[stats.norm, stats.laplace]).show()

## A family of noise distributions

Our mechanism releases a single number: the value of one record (0 when the target is absent, 1 when present) plus noise. We will look at several candidate noise distributions. For each we compare

$$H_0:\ X \sim \mathcal{D}(\mu_0=0,\ \text{scale}=s), \qquad H_1:\ X \sim \mathcal{D}(\mu_1=1,\ \text{scale}=s),$$

i.e. a **sensitivity** of $\Delta = \mu_1 - \mu_0 = 1$.

For each distribution we write down three things and then look at its ROC curve:

- the **PDF** $f(x)$,
- the **CDF** $F(x)$,
- the **privacy loss** (log-probability ratio) $\;\mathcal{L}(x) = \ln \dfrac{f_1(x)}{f_0(x)}\;$, which by the Neyman–Pearson lemma is the optimal test statistic.

Each interactive plot below is fixed to a single distribution; drag the **Scale** slider (aligned with the PDF panel) to see how its PDFs (left) and ROC curve (right) change. Check **Compute ε** to answer the reverse question from the recap above: given a tolerance $\delta$ (log-scale slider aligned with the ROC panel, down to $10^{-15}$), what is the **smallest** $\varepsilon$ for which the ROC stays below the $(\varepsilon, \delta)$ bound? The plot finds the tightest slope $m = e^{\varepsilon}$ automatically and draws the bound line.

### Laplace — $\mathcal{D} = \mathrm{Laplace}(\mu, b)$

$$f(x) = \frac{1}{2b}\,e^{-|x-\mu|/b}, \qquad F(x) = \begin{cases} \tfrac{1}{2}\,e^{(x-\mu)/b} & x < \mu,\\[4pt] 1 - \tfrac{1}{2}\,e^{-(x-\mu)/b} & x \ge \mu. \end{cases}$$

$$\mathcal{L}(x) = \frac{|x-\mu_0| - |x-\mu_1|}{b} = \frac{|x| - |x-1|}{b}.$$

In [ ]:
TheoryROCVisualizer(distribution="Laplace", selectable_distribution=False)

### Gaussian — $\mathcal{D} = \mathcal{N}(\mu, \sigma^2)$

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\,e^{-(x-\mu)^2/(2\sigma^2)}, \qquad F(x) = \Phi\!\left(\frac{x-\mu}{\sigma}\right),$$

where $\Phi$ is the standard normal CDF (no elementary closed form).

$$\mathcal{L}(x) = \frac{(x-\mu_0)^2 - (x-\mu_1)^2}{2\sigma^2} = \frac{2x-1}{2\sigma^2}.$$

In [ ]:
TheoryROCVisualizer(distribution="Gaussian", selectable_distribution=False)

### Student-$t$ — $\mathcal{D} = t_\nu(\mu, s)$, here $\nu = 3$

$$f(x) = \frac{\Gamma\!\big(\tfrac{\nu+1}{2}\big)}{\sqrt{\nu\pi}\,s\,\Gamma\!\big(\tfrac{\nu}{2}\big)} \left(1 + \frac{1}{\nu}\Big(\frac{x-\mu}{s}\Big)^2\right)^{-\frac{\nu+1}{2}}.$$

The CDF has no elementary form (it is the regularized incomplete beta function $I_{\,\cdot}\!\big(\tfrac{\nu}{2},\tfrac{\nu}{2}\big)$).

$$\mathcal{L}(x) = \frac{\nu+1}{2}\,\ln \frac{\nu + \big((x-\mu_0)/s\big)^2}{\nu + \big((x-\mu_1)/s\big)^2}.$$

In [ ]:
TheoryROCVisualizer(distribution="Student t", selectable_distribution=False)

### Cauchy — $\mathcal{D} = \mathrm{Cauchy}(\mu, \gamma)$

$$f(x) = \frac{1}{\pi\gamma\left(1 + \big((x-\mu)/\gamma\big)^2\right)}, \qquad F(x) = \frac{1}{2} + \frac{1}{\pi}\arctan\!\Big(\frac{x-\mu}{\gamma}\Big).$$

$$\mathcal{L}(x) = \ln \frac{\gamma^2 + (x-\mu_0)^2}{\gamma^2 + (x-\mu_1)^2}.$$

The likelihood ratio is **not monotone** in $x$ (it tends to $1$ in both tails), so the optimal test rejects on an *interval*, not a one-sided threshold. The ROC is obtained by sorting outcomes by their log-likelihood ratio, which makes it concave.

In [ ]:
TheoryROCVisualizer(distribution="Cauchy", selectable_distribution=False)

### Logistic — $\mathcal{D} = \mathrm{Logistic}(\mu, s)$

$$f(x) = \frac{e^{-(x-\mu)/s}}{s\left(1 + e^{-(x-\mu)/s}\right)^2}, \qquad F(x) = \frac{1}{1 + e^{-(x-\mu)/s}}.$$

$$\mathcal{L}(x) = \frac{\mu_1 - \mu_0}{s} - 2\ln \frac{1 + e^{-(x-\mu_1)/s}}{1 + e^{-(x-\mu_0)/s}}.$$

In [ ]:
TheoryROCVisualizer(distribution="Logistic", selectable_distribution=False)

## A distribution with no closed form: the difference of two lognormals

So far every candidate had a tidy PDF and CDF that we could differentiate, integrate, and plug into $\mathcal{L}$. That is the exception, not the rule. Consider noise built from **lognormal** random variables.

A positive random variable $Y$ is **lognormal** if its logarithm is normal:

$$\ln Y \sim \mathcal{N}(0,\, s^2) \qquad\Longleftrightarrow\qquad Y = e^{s Z},\ \ Z \sim \mathcal{N}(0,1).$$

Lognormals are positive and right-skewed: most of the mass sits near $1$, with a long heavy tail to the right that grows as $s$ increases.

Our noise is the **difference of two independent lognormals**,

$$N = A - B, \qquad A, B \stackrel{\text{i.i.d.}}{\sim} \mathrm{LogNormal}(0,\, s),$$

which is symmetric about $0$ (so it is an unbiased noise) and heavy-tailed. The mechanism then releases $\mu + N$, with $\mu = 0$ when the target is absent and $\mu = 1$ when present.

The catch: the difference of two lognormals has **no known closed-form PDF or CDF**. We therefore cannot write $f$, $F$, or $\mathcal{L}$ down, and we cannot draw the theoretical ROC the way we did above. We need another tool.

## Estimating the ROC by sampling

When we cannot integrate the densities, we can still **simulate**. This is the core trick behind empirical auditing:

1. Draw many samples of the negative output ($\mu = 0$) and many of the positive output ($\mu = 1$).
2. Approximate the densities $f_0, f_1$ by **histograms** of those samples.
3. Sweep a threshold over the samples and read off $(\widehat{\mathrm{FPR}}, \widehat{\mathrm{TPR}})$ at each level — this traces an **empirical ROC** curve (here computed with scikit-learn's `roc_curve`).

The plot below applies this to the lognormal-difference noise. There is **no theory curve to compare against** — the dashed empirical ROC is all we have. Press **Generate Samples** to redraw a fresh batch, and increase **Samples / class** to see the estimate steady. Check **Compute ε** to apply the same $\delta \to \varepsilon$ construction to the **sampled** ROC — exactly what an auditor can compute without knowing the true density (initial $\delta = 10^{-2}$).

In [ ]:
samples_per_class = 500
EmpROCVisualizer(samples_per_class, distribution="Lognormal difference", selectable_distribution=False)

## One explorer for all of them

Finally, here is the empirical explorer over **all** the noise distributions we have met, now driven by sampling. Pick a distribution from the dropdown:

- for the analytic ones it overlays the **sampled** estimate (dashed, with histograms) on the **exact** theory curve, so you can watch the estimate converge as you raise *Samples / class*;
- for the lognormal difference only the sampled estimate exists.

This is the bridge to auditing: in both cases we never needed a formula — only the ability to *run the mechanism and count*.

In [ ]:
EmpROCVisualizer(samples_per_class)

## From exploring ROCs to auditing a mechanism

A real auditor faces a **black box**: some implementation of a supposedly private mechanism, with no formula attached. Playing the adversary, we want to turn our sampling power into a single number — a **lower bound** on the mechanism's $\varepsilon$. The recipe specializes the sampling idea above:

1. Run the mechanism many times on the **negative** dataset and many times on the **positive** (neighboring) dataset.
2. Apply a fixed guessing rule, e.g. "guess positive if the output exceeds a threshold $\tau$".
3. Count the empirical rates
$$\widehat{\mathrm{FPR}} = \frac{\#\{\text{positive guesses on negatives}\}}{n}, \qquad \widehat{\mathrm{TPR}} = \frac{\#\{\text{positive guesses on positives}\}}{n}.$$
4. Convert to an empirical privacy lower bound
$$\widehat{\varepsilon} = \ln \frac{\widehat{\mathrm{TPR}} - \delta}{\widehat{\mathrm{FPR}}}.$$

## Does a single run give the *exact* value?

No. Each $\widehat{\varepsilon}$ is computed from finitely many coin flips, so it is itself a **random variable**. Run the same audit a few times and you get a different number every time — sometimes *above* the true $\varepsilon$, sometimes below.

That asymmetry is dangerous. We want $\widehat\varepsilon$ to be a **certified lower bound** ("the mechanism leaks *at least* this much"). A naive estimate that plugs the raw counts into the formula overshoots the truth roughly half the time, which would let us "prove" more privacy loss than really exists — a false alarm. Let's see the wobble directly.

In [ ]:
# A small, fixed auditing setup: sum of a 5-record dataset released with Laplace noise.
scale = 1
num_experiments = 1000
threshold = 0.5
delta = 0
dataset_neg = [0, 0, 0, 0, 0]
dataset_pos = [0, 0, 0, 0, 1]

# The true epsilon for this threshold rule (we can compute it because we know the distributions).
dist_neg = stats.laplace(np.sum(dataset_neg), scale)
dist_pos = stats.laplace(np.sum(dataset_pos), scale)
true_FPR = FPR_from_threshold(dist_neg, threshold)
true_TPR = TPR_from_threshold(dist_pos, threshold)
true_eps = np.log((true_TPR - delta) / true_FPR)

mechanism = lambda x: Laplace_sum(x, scale)
guesser = lambda x: threshold_guesser(x, threshold)

# Naive estimate: error_prob = -1 means "use the raw empirical FPR/TPR, no correction".
print(f"True epsilon: {true_eps:.4f}\n")
print("Five independent naive audits (they scatter around -- and sometimes exceed -- the truth):")
for i in range(5):
    eps_hat = epsilon_lower_bound(
        mechanism, dataset_neg, dataset_pos, guesser, num_experiments, -1, delta
    )
    print(f"  run {i + 1}: epsilon_hat = {eps_hat:.4f}")

## Tail bounds: from a point estimate to a *valid* lower bound

To make $\widehat\varepsilon$ trustworthy we must not use the raw empirical rate. Instead we replace each rate by a **confidence bound**: a value such that, with probability at least $1-\alpha$, the true rate lies on the safe side of it. Concretely, to lower-bound $\varepsilon = \ln\frac{\mathrm{TPR}-\delta}{\mathrm{FPR}}$ we use a high-confidence **lower** bound on the TPR and a high-confidence **upper** bound on the FPR.

How tight can such confidence bounds be? Each count is a sum of $n$ Bernoulli trials, $\mathrm{Binomial}(n, p)$, and the whole game is controlling how far the empirical mean strays from $p$. That is exactly what **concentration / tail inequalities** quantify. Two complementary views:

- **"How big an error $\alpha$ must I tolerate for a fixed confidence?"** — Markov, Chebyshev, Hoeffding.
- **"How small a failure probability $\beta$ do I get for a fixed error $\alpha$?"** — Chebyshev, Hoeffding, Chernoff.

The first explorer fixes the confidence level (via $\log\beta$) and shows the error $\alpha$ each bound guarantees as the sample size $n$ grows. Markov barely moves, Chebyshev decays like $1/\sqrt{n}$, and Hoeffding (sub-Gaussian) is tightest.

In [ ]:
binomial_alpha_bounds_interactive()

Flipping the question: fix the error width $\alpha$ and ask how quickly the **failure probability** $\beta$ shrinks with $n$. Here the multiplicative **Chernoff** bound shines — it decays *exponentially* in $n$, far faster than Chebyshev's polynomial rate. This exponential decay is why a few thousand auditing runs already pin the rates down tightly enough to certify $\varepsilon$. (The $y$-axis is logarithmic, so exponential decay shows up as a straight line.)

In [ ]:
binomial_beta_bounds_interactive(log_y=True)

## Putting it together: naive vs. safe estimates

The auditing helper `epsilon_lower_bound` implements both options:

- **Naive** (`error_prob = -1`): plug the raw empirical FPR/TPR straight into the formula. Centered near the truth, but overshoots about half the time — *not* a valid lower bound.
- **Safe** (`error_prob = ` $\alpha$): replace the rates with **Clopper–Pearson** confidence limits — exact Binomial intervals computed from the Beta distribution, the inverse view of the same tail bounds above — so that with probability $\ge 1-\alpha$ the reported $\widehat\varepsilon$ does **not** exceed the true $\varepsilon$.

The histogram below repeats the audit 200 times for each method. The safe distribution sits deliberately to the **left** of the true $\varepsilon$ (red line): it trades a little tightness for the guarantee that it (almost) never overshoots — exactly what a certified privacy lower bound requires.

In [ ]:
prob = 0.05  # alpha: allowed failure probability for the safe (Clopper-Pearson) bound

naive_eps_arr = [
    epsilon_lower_bound(mechanism, dataset_neg, dataset_pos, guesser, num_experiments, -1, delta)
    for _ in range(200)
]
safe_eps_arr = [
    epsilon_lower_bound(mechanism, dataset_neg, dataset_pos, guesser, num_experiments, prob, delta)
    for _ in range(200)
]

# Plot the histogram of the naive and safe epsilon and add a vertical line for the true epsilon.
plt.hist(naive_eps_arr, bins=50, alpha=0.5, label='Naive')
plt.hist(safe_eps_arr, bins=50, alpha=0.5, label='Safe')
plt.axvline(x=true_eps, color='red', label='True')
plt.legend()
plt.title('Histogram of Naive and Safe Epsilon Estimates')
plt.xlabel('Epsilon')
plt.ylabel('Frequency')
plt.show()

## Animating the threshold sweep

The same plots can be made *playable*. The frames are produced by a build sidecar (`animations/threshold-sweep.py`) and combined into the player below while the site renders — the post itself imports nothing. Step through frame-by-frame with the **Prev / Next** buttons, or press **play** to run it at the configured frame rate. Sweeping the decision threshold $\tau$ traces out the ROC curve.

<iframe src="../../../_generated/animations/blog-posts/privacy-auditing/threshold-sweep.html" width="100%" height="430" style="border:0" loading="lazy" title="Threshold sweep tracing the ROC curve" onload="this.style.height=this.contentWindow.document.documentElement.scrollHeight+'px'"></iframe>